# Batch optotagging metric generation

Generate the per-probe `*_laser_response_metrics.csv` files for a **list of
sessions**. This is the batch equivalent of Steps 1–4 of
`optotagging_Anna_nwb.ipynb`: for each session it loads the ephys NWB, locates the
raw `ecephys_clipped` asset (NIDAQ onsets + `*opto.csv`), computes laser-response
metrics for every QC unit on each stimulated probe, and writes one CSV per probe
to `SAVE_FOLDER`.

Once the CSVs exist, use `optotagging_Anna_nwb.ipynb` (Step 6, *Option A*) to append
the results back onto each NWB and select tagged units. Failures are caught
per session so one bad session does not stop the batch.

## Setup

In [ ]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd

from nwb_utils import NWBUtils
from optotagging_Anna_nwb import (
    OptotaggingAnalysisNWB,
    find_recording_clipped_folder,
)

print(f"✅ Modules loaded from: {MODULE_PATH}")

## Configuration

List the sessions to process and set the NIDAQ / output parameters (same meaning
as in the single-session notebook).

In [ ]:
SESSION_NAMES = [
    "ecephys_839480_2026-06-03_15-09-14_sorted-bandpass_2026-07-18_00-07-17",
    # add more sorted session names here...
]

SAVE_FOLDER = "/root/capsule/scratch/opto_tagging_Anna"

LASER_EVENT_ID = "2"   # NIDAQ channel-2 digital-input label
OPTO_RECORDING = 0     # segment index with the laser stimulation
FLIP_NIDAQ = False     # subtract 0.5 s if the sync signal was flipped
PRE_OPTO_DURATION = None  # set to a float (s) to compute pre-stim ISI / rate

print(f"{len(SESSION_NAMES)} session(s) to process -> {SAVE_FOLDER}")

## Per-session metric generation

The helper loads one session, computes metrics for every stimulated probe, saves
the CSVs, and always closes the NWB IO handle.

In [ ]:
def generate_metrics_for_session(
    session_name,
    save_folder,
    laser_event_id="2",
    opto_recording=0,
    flip_nidaq=False,
    pre_opto_duration=None,
):
    """Compute and save per-probe laser-response metric CSVs for one session."""
    nwb_data = NWBUtils.read_ephys_nwb(session_name=session_name)
    if nwb_data is None:
        raise RuntimeError(f"Failed to load ephys NWB for '{session_name}'.")

    try:
        clipped = find_recording_clipped_folder(session_name)
        analysis = OptotaggingAnalysisNWB(
            nwb_data=nwb_data,
            session_name=session_name,
            recording_clipped_folder=clipped,
            laser_event_id=laser_event_id,
            opto_recording=opto_recording,
            flip_NIDAQ=flip_nidaq,
        )

        if "type" in analysis.trial_ids.columns:
            trial_types = list(np.unique(analysis.trial_ids["type"]))
        else:
            trial_types = ["all"]
        powers = (
            list(np.unique(analysis.trial_ids["power"]))
            if "power" in analysis.trial_ids.columns
            else [None]
        )
        trials_query = {"type": trial_types, "power": powers}
        suffixes = [None, "mW"]

        Path(save_folder).mkdir(parents=True, exist_ok=True)
        saved = []
        for probe in analysis.get_stream_names():
            metrics = analysis.one_probe_laser_responses(
                trials_query=trials_query,
                probe=probe,
                suffixes=suffixes,
                ignore_onset_offset=True,
                pre_opto_duration=pre_opto_duration,
            )
            if len(metrics) == 0:
                continue
            metrics = OptotaggingAnalysisNWB.add_best_power_columns(metrics, trial_types)
            out_csv = Path(save_folder) / f"{analysis.session}_{probe}_laser_response_metrics.csv"
            metrics.to_csv(out_csv, index=False)
            saved.append(str(out_csv))
            print(f"    saved {out_csv.name} ({len(metrics)} units)")

        return {
            "session": session_name,
            "n_qc_units": int(len(analysis.qc_units)),
            "n_onsets": int(len(analysis.laser_onset_times)),
            "trial_types": trial_types,
            "n_csvs": len(saved),
        }
    finally:
        if hasattr(nwb_data, "io"):
            nwb_data.io.close()

## Run the batch

In [ ]:
summary = []
for session_name in SESSION_NAMES:
    print(f"\n=== {session_name} ===")
    try:
        info = generate_metrics_for_session(
            session_name,
            SAVE_FOLDER,
            laser_event_id=LASER_EVENT_ID,
            opto_recording=OPTO_RECORDING,
            flip_nidaq=FLIP_NIDAQ,
            pre_opto_duration=PRE_OPTO_DURATION,
        )
        info["status"] = "ok"
        print(
            f"  QC units: {info['n_qc_units']}, onsets: {info['n_onsets']}, "
            f"CSVs written: {info['n_csvs']}"
        )
    except Exception as exc:  # noqa: BLE001 - keep the batch going
        info = {"session": session_name, "status": f"ERROR: {exc}"}
        print(f"  ERROR: {exc}")
    summary.append(info)

summary_df = pd.DataFrame(summary)
summary_df

## Summary

`summary_df` lists each session's status, QC-unit count, laser-onset count and the
number of CSVs written. Rows with a non-`ok` status hit an error (missing raw
asset, NIDAQ mismatch, etc.) and were skipped.

In [ ]:
ok = summary_df[summary_df["status"] == "ok"] if len(summary_df) else summary_df
failed = summary_df[summary_df["status"] != "ok"] if len(summary_df) else summary_df
print(f"Succeeded: {len(ok)} / {len(summary_df)} sessions")
if len(failed):
    print("Failed sessions:")
    for _, r in failed.iterrows():
        print(f"  {r['session']}: {r['status']}")